In [1]:

import nltk
import random
import numpy as np
import pandas as pd
import pprint, time
from sklearn.model_selection import KFold
!curl -O https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_train.txt
!curl -O https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_test.txt
with open("GSD_train.txt", encoding='utf-8') as f:
    data = f.read()
sent = data.split('\n\n')
for i in range(3):
    print(sent[i])
    print('')
for i in sent[-5:]:
    print(i, '\n')
tokens = []
for sentence in sent:
    s = []
    for word in sentence.split('\n'):
        if word:
            token = (word.split()[1], word.split()[3])
            s.append(token)
    tokens.append(s)
print(tokens[0:10])
print(len(sent))


def word_given_tag(word, tag, train_bag):
    tag_list = [pair for pair in train_bag if pair[1] == tag]
    count_tag = len(tag_list)
    w_given_tag_list = [pair[0] for pair in tag_list if pair[0] == word]
    count_w_given_tag = len(w_given_tag_list)
    return (count_w_given_tag, count_tag)


def t2_given_t1(t2, t1, train_bag):
    tags = [pair[1] for pair in train_bag]
    count_t1 = len([t for t in tags if t == t1])
    count_t2_t1 = 0
    for index in range(len(tags) - 1):
        if tags[index] == t1 and tags[index + 1] == t2:
            count_t2_t1 += 1
    return (count_t2_t1, count_t1)


# KFold
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=101)
accuracies = []  # точности для фолда

for fold, (train_index, test_index) in enumerate(kf.split(tokens)):
    train_set = [tokens[i] for i in train_index]
    test_set = [tokens[i] for i in test_index]

    train_tagged_words = [tup for sent in train_set for tup in sent]
    test_tagged_words = [tup for sent in test_set for tup in sent]

    tags = {tag for word, tag in train_tagged_words}
    vocab = {word for word, tag in train_tagged_words}

    tags_matrix = np.zeros((len(tags), len(tags)), dtype='float32')
    for i, t1 in enumerate(list(tags)):
        for j, t2 in enumerate(list(tags)):
            tags_matrix[i, j] = t2_given_t1(t2, t1, train_tagged_words)[0] / t2_given_t1(t2, t1, train_tagged_words)[1]

    tags_df = pd.DataFrame(tags_matrix, columns=list(tags), index=list(tags))

    random.seed(1234)
    rndom = [random.randint(1, len(test_set)) for x in range(10)]
    test_run = [test_set[i] for i in rndom]
    test_run_base = [tup for sent in test_run for tup in sent]
    test_words = [tup[0] for sent in test_run for tup in sent]


    def Viterbi(words, train_bag=train_tagged_words):
        state = []
        T = list(set([pair[1] for pair in train_bag]))

        for key, word in enumerate(words):
            p = []
            for tag in T:
                if key == 0:
                    transition_p = tags_df.loc['PUNCT', tag]
                else:
                    transition_p = tags_df.loc[state[-1], tag]

                emission_p = word_given_tag(word, tag, train_bag)[0] / word_given_tag(word, tag, train_bag)[1]
                state_probability = emission_p * transition_p
                p.append(state_probability)

            pmax = max(p)
            state_max = T[p.index(pmax)]
            state.append(state_max)
        return list(zip(words, state))


    start = time.time()
    tagged_seq = Viterbi(test_words)
    end = time.time()
    difference = end - start
    print(f"Fold {fold + 1}: Время выполнения в секундах: {difference}")

    check = [i for i, j in zip(tagged_seq, test_run_base) if i == j]
    accuracy = len(check) / len(tagged_seq)
    accuracies.append(accuracy)
    print(f"Fold {fold + 1}: Точность алгоритма Витерби, %: {accuracy * 100}")

mean_accuracy = np.mean(accuracies)
print(f"Средняя точность по всем фолдам: {mean_accuracy * 100}%")


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 7448k  100 7448k    0     0  3644k      0  0:00:02  0:00:02 --:--:-- 3645k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 81386  100 81386    0     0   115k      0 --:--:-- --:--:-- --:--:--  115k
1	Начальный	начальный	ADJ	JJL	Case=Nom|Degree=Pos|Gender=Masc|Number=Sing	2	amod	_	_
2	ролик	ролик	NOUN	NN	Animacy=Inan|Case=Nom|Gender=Masc|Number=Sing	17	nsubj	_	SpaceAfter=No
3	,	,	PUNCT	,	_	5	punct	_	_
4	или	или	CCONJ	CC	_	5	cc	_	_
5	опенинг	опенинг	NOUN	NN	Animacy=Inan|Case=Nom|Gender=Masc|Number=Sing	2	conj	_	_
6	(	(	PUNCT	(	_	7	punct	_	SpaceAfter=No
7	от	от	ADP	IN	_	5	parataxis	_	SpaceAfter=No
8	,	,	PUNCT